In [1]:
# %pip install --user pyfaidx

In [1]:
import pandas as pd
from pyfaidx import Fasta

In [2]:
archivo_genoma = "/home/jbs1009/TFM/datosGene4PD/Homo_sapiens.GRCh37.completo.fa"
datos_clinvar = "/home/jbs1009/TFM/datosGene4PD/variant_summary.txt.gz"
base_procesada = "/home/jbs1009/TFM/datosGene4PD/base_procesada_OR_709.csv"

In [3]:
df_base_procesada = pd.read_csv(base_procesada)

In [4]:
df_base_procesada

,Chr,gene_symbol,SNPs_symbol,SNP_position,effect_allele,alternate_allele,joint_phase_OR,Chr_corr,Gene_symbol_corr,SNP_position_corr,Effect_allele_corr,Alternate_allele_corr,Cadena,Valida_alelos
0,6,GPR126,rs757765789,142758601,T,G,1.06,6,ADGRG6,142758601,T,['G'],+,Coincide perfecto
1,12,SLC2A13,rs1994090,40428561,G,T,12.05,12,SLC2A13,40428561,G,"['A', 'T', 'C']",-,Coincide perfecto
2,12,SLC2A13,rs2708453,40478652,G,T,12.05,12,SLC2A13,40478652,G,"['A', 'T']",-,Coincide perfecto
3,12,SLC2A13,rs4768212,40474147,C,T,12.05,12,SLC2A13,40474147,C,"['A', 'T']",-,Coincide perfecto
4,14,SLC2A15,rs7304281,40465942,T,C,12.05,12,LINC02347,126945694,T,"['G', 'C']",+,Coincide perfecto
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
704,17,BRIP1,rs61169879,59917366,T,C,1.09,17,BRIP1,59917366,C,"['A', 'T']",-,Coincide perfecto
705,17,DNAH17,rs666463,76425480,A,T,1.08,17,DNAH17,76425480,A,['T'],-,Coincide perfecto
706,18,ASXL3,rs1941685,31304318,T,G,1.05,18,ASXL3,31304318,G,"['T', 'C']",+,Coincide perfecto
707,20,CRLS1,rs77351827,6006041,T,C,1.08,20,CRLS1,6006041,C,['T'],+,Coincide perfecto


In [5]:
gene_symbols = df_base_procesada["Gene_symbol_corr"].dropna().unique().tolist()

In [6]:
genoma = Fasta(archivo_genoma)

In [7]:
columnas = ["Assembly", "Type", "ClinicalSignificance", "RS# (dbSNP)", "GeneSymbol", "Chromosome", "PositionVCF", "ReferenceAllele", "AlternateAllele"]
df_clinvar = pd.read_csv(datos_clinvar, sep = "\t", compression = "gzip", usecols = columnas, low_memory = False)

In [8]:
df_clinvar

,Type,GeneSymbol,ClinicalSignificance,RS# (dbSNP),Assembly,Chromosome,ReferenceAllele,AlternateAllele,PositionVCF
0,Indel,AP5Z1,Pathogenic/Likely pathogenic,397704705,GRCh37,7,na,na,4820844
1,Indel,AP5Z1,Pathogenic/Likely pathogenic,397704705,GRCh38,7,na,na,4781213
2,Deletion,AP5Z1,Pathogenic,397704709,GRCh37,7,na,na,4827360
3,Deletion,AP5Z1,Pathogenic,397704709,GRCh38,7,na,na,4787729
4,single nucleotide variant,ZNF592,Uncertain significance,150829393,GRCh37,15,na,na,85342440
...,...,...,...,...,...,...,...,...,...
8981168,single nucleotide variant,GJB4,Likely benign,-1,GRCh38,1,na,na,34761922
8981169,single nucleotide variant,XPO1,Likely benign,-1,GRCh37,2,na,na,61726932
8981170,single nucleotide variant,XPO1,Likely benign,-1,GRCh38,2,na,na,61499797
8981171,single nucleotide variant,EP300,Likely benign,-1,GRCh37,22,na,na,41573839


In [9]:
filtros = ((df_clinvar["Assembly"] == "GRCh37") & (df_clinvar["Type"] == "single nucleotide variant") & (df_clinvar["ClinicalSignificance"] == "Benign") & (df_clinvar["GeneSymbol"].isin(gene_symbols)) & (df_clinvar["RS# (dbSNP)"] != -1))

In [10]:
df_clinvar_filtrado = df_clinvar[filtros].copy()

In [15]:
df_clinvar_filtrado.reset_index(drop = False).drop("index", axis = 1)

,Type,GeneSymbol,ClinicalSignificance,RS# (dbSNP),Assembly,Chromosome,ReferenceAllele,AlternateAllele,PositionVCF
0,single nucleotide variant,RAI1,Benign,104894633,GRCh37,17,na,na,17701685
1,single nucleotide variant,TAP2,Benign,241447,GRCh37,6,na,na,32796751
2,single nucleotide variant,TAP2,Benign,241448,GRCh37,6,na,na,32796685
3,single nucleotide variant,ITIH1,Benign,678,GRCh37,3,na,na,52820981
4,single nucleotide variant,SERPINA1,Benign,6647,GRCh37,14,na,na,94847415
...,...,...,...,...,...,...,...,...,...
4920,single nucleotide variant,SCARB2,Benign,7676834,GRCh37,4,na,na,77101637
4921,single nucleotide variant,ANK2,Benign,17483231,GRCh37,4,na,na,114299347
4922,single nucleotide variant,ANK2,Benign,7689214,GRCh37,4,na,na,114067145
4923,single nucleotide variant,ANK2,Benign,3112980,GRCh37,4,na,na,114067139


In [16]:
df_clinvar_filtrado.to_csv("/home/jbs1009/TFM/datosGene4PD/datos_clinvar_filtrados.csv", index = False)